# StateSet Agents v1.0 GA — 1-Click GPU + Triton + River AI 0.11 Evidence Capture (Colab Ultra)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/v1_ga_gpu_river_evidence_colab.ipynb)

> **Turnkey `v1.0.0` GA Evidence Capture for Google AI Ultra (H100 / A100 / L4) + River AI `0.11`.**
>
> Run **Runtime → Run all** to generate SHA-256 attested `v1_ga_evidence.json` and `v1_ga_evidence.md` tables ready to paste directly into the `v1.0.0` GitHub Release Notes, `CHANGELOG.md`, and `docs/PROOFS.md`:
> - Real CUDA `torch.cuda.max_memory_allocated()` comparison (Full Logits vs. Chunked / Triton SRAM kernel)
> - Deterministic 3-seed Open-Weight RL Ablation + 6-Domain Enterprise Verifiable Sandboxes
> - Real HuggingFace Open-Weight Qwen LoRA Step-0 CAA + Chunked GSPO training run
> - Optional live River AI `0.11` preflight + server capabilities check (reads `RIVER_API_KEY` from Colab Secrets)

## 1. Install `stateset-agents[training,api]` & Inspect Runtime

In [ ]:
import os
import subprocess
import sys

INSTALL_SPEC = os.environ.get(
    "STATESET_INSTALL_SPEC",
    "stateset-agents[training,api] @ git+https://github.com/stateset/stateset-agents.git@release/0.99.1",
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", INSTALL_SPEC])

from stateset_agents.utils.colab import detect_colab_runtime
rt = detect_colab_runtime(probe_cuda=True)
print("Detected Runtime:", rt["gpu_name"], f"({rt['vram_gb']} GB)", "Triton:", rt["triton_available"])

## 2. Execute Packaged GPU + River Evidence Capture (`capture_v1_evidence`)

In [ ]:
from pathlib import Path
from stateset_agents.utils.colab import (
    capture_v1_evidence,
    get_colab_secret,
    is_colab_runtime,
    render_v1_evidence_markdown,
)

# Configure your capture run
HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if rt["cuda_available"] else ""
HF_STEPS = 6 if rt["cuda_available"] else 2
RIVER_KEY = get_colab_secret("RIVER_API_KEY")
RIVER_MODE = "preflight" if (RIVER_KEY and rt["river_python_compatible"]) else "offline"

OUT_DIR = Path("outputs/v1_ga_evidence")
evidence = capture_v1_evidence(
    output_dir=OUT_DIR,
    hf_model=HF_MODEL or None,
    hf_steps=HF_STEPS,
    skip_hf_model=not bool(HF_MODEL),
    river_mode=RIVER_MODE,
)

md_report = render_v1_evidence_markdown(evidence)
print(md_report)

## 3. Download `v1_ga_evidence.json` & `v1_ga_evidence.md`

In [ ]:
import zipfile
from pathlib import Path
from stateset_agents.utils.colab import is_colab_runtime

bundle = Path("outputs/v1_ga_evidence_bundle.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in ("v1_ga_evidence.json", "v1_ga_evidence.md"):
        fpath = OUT_DIR / fname
        if fpath.exists():
            zf.write(fpath, arcname=fname)

print("Bundled GA evidence:", bundle, f"({bundle.stat().st_size} bytes)")
if is_colab_runtime():
    from google.colab import files
    files.download(str(bundle))